In [ ]:
# ---------------------------------------------------------------
# Importações e configuração
# ---------------------------------------------------------------

import pandas as pd
import jinja2
from pathlib import Path #Path monta caminhos que funcionam em Windows, Mac e Linux

# Mostra todas as colunas ao exibir um DataFrame (o padrão do pandas esconde as do meio)
pd.set_option("display.max_columns", None)

# Caminho do CSV
CAMINHO_CSV = Path.cwd() / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# ---------------------------------------------------------------
# Verificação de existência do arquivo
# ---------------------------------------------------------------
if not CAMINHO_CSV.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {CAMINHO_CSV.resolve()}\n"
        "Baixe o CSV no Kaggle, salve na pasta 'data/' e confira o nome do arquivo.\n"
        "Também confira se o notebook foi aberto a partir da pasta raiz do projeto."
    )

print("Arquivo encontrado:", CAMINHO_CSV.resolve())

# ---------------------------------------------------------------
# Leitura do CSV -> DataFrame BRUTO
# ---------------------------------------------------------------
df_telco = pd.read_csv(CAMINHO_CSV)
print("Dimensões:", df_telco.shape)
# ESPERADO (base clássica do Kaggle): (7043, 21)


# ---------------------------------------------------------------
# Primeiras linhas: para ter uma noção de como os valores se parecem
# ---------------------------------------------------------------
display(df_telco.head())

# ---------------------------------------------------------------
# Tipos de dados e contagem de valores não nulos
# ---------------------------------------------------------------
df_telco.info()
# ESPERADO: TotalCharges aparece como TEXTO (object ou str), embora represente dinheiro.

# ---------------------------------------------------------------
# Resumo estatístico de TODAS as colunas
# ---------------------------------------------------------------
display(df_telco.describe(include="all").T)

Arquivo encontrado: C:\ProjetoPortifolio\customer-churn-analytics\data\WA_Fn-UseC_-Telco-Customer-Churn.csv
Dimensões: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ---------------------------------------------------------------
# Valores distintos de cada coluna categórica
# ---------------------------------------------------------------
# select_dtypes(exclude="number") pega todas as colunas que NÃO são numéricas.
# Removemos customerID (tem um valor por cliente) e TotalCharges (é número disfarçado de texto).
colunas_categoricas = (
    df_telco.select_dtypes(exclude="number")
    .columns.drop(["customerID", "TotalCharges"], errors="ignore")
)

for coluna in colunas_categoricas:
    print(f"\n--- {coluna} ---")
    # dropna=False -> se existir valor nulo, ele aparece na contagem em vez de ser escondido
    print(df_telco[coluna].value_counts(dropna=False).to_string())

# ESPERADO: poucas categorias por coluna, sem variações de grafia.


--- gender ---
gender
Male      3555
Female    3488

--- Partner ---
Partner
No     3641
Yes    3402

--- Dependents ---
Dependents
No     4933
Yes    2110

--- PhoneService ---
PhoneService
Yes    6361
No      682

--- MultipleLines ---
MultipleLines
No                  3390
Yes                 2971
No phone service     682

--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526

--- OnlineSecurity ---
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526

--- OnlineBackup ---
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526

--- DeviceProtection ---
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526

--- TechSupport ---
TechSupport
No                     3473
Yes                    2044
No internet service    1526

--- StreamingTV ---
StreamingTV
No                     2810
Yes                    2

In [ ]:
# ---------------------------------------------------------------
# Espaços sobrando no começo/fim dos textos
# ---------------------------------------------------------------
colunas_texto = df_telco.select_dtypes(exclude="number").columns

# Para cada coluna de texto, compara o valor com a versão sem espaços nas pontas.
# Se forem diferentes, há espaço sobrando.
espacos_sobrando = {
    col: int((df_telco[col] != df_telco[col].str.strip()).sum())
    for col in colunas_texto
}
espacos_sobrando = {col: n for col, n in espacos_sobrando.items() if n > 0}

print("Colunas com espaços nas pontas:", espacos_sobrando if espacos_sobrando else "nenhuma")

# ESPERADO: provavelmente só TotalCharges (os valores em branco " " contam como espaço).

Colunas com espaços nas pontas: {'TotalCharges': 11}


In [ ]:
# ---------------------------------------------------------------
# Duplicatas
# ---------------------------------------------------------------
linhas_duplicadas = df_telco.duplicated().sum()          # linhas 100% idênticas
ids_repetidos = df_telco["customerID"].duplicated().sum() # mesmo cliente aparecendo mais de uma vez

print("Linhas totalmente duplicadas:", linhas_duplicadas)
print("customerID repetidos:", ids_repetidos)

# até que você decida o tratamento e registre no "Registro de decisões").
if linhas_duplicadas > 0 or ids_repetidos > 0:
    print("⚠️ Há duplicatas! Inspecione-as antes de continuar, por exemplo:")
    print("   df_telco[df_telco['customerID'].duplicated(keep=False)].sort_values('customerID')")

#ESPERADO: 0 e 0.

Linhas totalmente duplicadas: 0
customerID repetidos: 0


In [ ]:
# ---------------------------------------------------------------
# Valores ausentes: nulos "formais" e células "em branco"
# ---------------------------------------------------------------
# 1) Nulos formais (NaN) — é o que o .isna() enxerga.
nulos_formais = df_telco.isna().sum()
print("Nulos formais (NaN) por coluna, somente colunas com problema:")
print(nulos_formais[nulos_formais > 0] if (nulos_formais > 0).any() else "nenhum")

# 2) Células "em branco": texto que contém só espaços (ex.: " ").
# o .isna() NÃO enxerga essas células, pois para o pandas " " é um texto válido.
# Por isso a checagem acima sozinha daria a falsa impressão de dado limpo.
brancos = df_telco[colunas_texto].apply(lambda coluna: coluna.str.strip().eq("")).sum()
print("\nCélulas em branco (só espaços) por coluna, somente colunas com problema:")
print(brancos[brancos > 0] if (brancos > 0).any() else "nenhuma")

# ESPERADO: nenhum nulo formal; 11 células em branco em TotalCharges.

Nulos formais (NaN) por coluna, somente colunas com problema:
nenhum

Células em branco (só espaços) por coluna, somente colunas com problema:
TotalCharges    11
dtype: int64


In [ ]:
# ---------------------------------------------------------------
# Cópia de trabalho
# ---------------------------------------------------------------
# Serve pra manter o dado bruto sem alterações.
df_telco_limpo = df_telco.copy()

# Remove espaços nas pontas de todas as colunas de texto para diferenciar strings.
for coluna in colunas_texto:
    df_telco_limpo[coluna] = df_telco_limpo[coluna].str.strip()

# Nesta etapa o " " de TotalCharges virou "" (texto vazio). Isso será tratado a seguir.

In [ ]:
# ---------------------------------------------------------------
# Conversão de TotalCharges (texto -> número)
# ---------------------------------------------------------------
# Quantos valores "vazios" existiam ANTES da conversão
# (texto vazio "" que veio dos espaços em branco, ou nulo de fato).
vazios_antes = int(
    (df_telco_limpo["TotalCharges"] == "").sum() + df_telco_limpo["TotalCharges"].isna().sum()
)

# pd.to_numeric converte texto em número.
# errors="coerce" -> o que não puder ser convertido vira NaN (em vez de gerar um erro).
df_telco_limpo["TotalCharges"] = pd.to_numeric(df_telco_limpo["TotalCharges"], errors="coerce")

# Quantos NaN existem DEPOIS da conversão
nulos_depois = int(df_telco_limpo["TotalCharges"].isna().sum())

print("Vazios antes da conversão :", vazios_antes)
print("Nulos depois da conversão :", nulos_depois)
print("Novo tipo de TotalCharges :", df_telco_limpo["TotalCharges"].dtype)

# ATENÇÃO: errors="coerce" é "silencioso": QUALQUER texto inválido (ex.: "abc") também viraria NaN
# sem avisar. Este assert é a trava de segurança: se os dois números diferirem, a conversão
# escondeu algum valor inesperado e é preciso investigar antes de continuar.
assert vazios_antes == nulos_depois, (
    "A conversão gerou mais nulos do que o esperado: há valores inválidos além dos espaços em branco."
)

#ESPERADO (base clássica): 11 e 11, e o tipo passa a ser float64.

Vazios antes da conversão : 11
Nulos depois da conversão : 11
Novo tipo de TotalCharges : float64


In [ ]:
# ---------------------------------------------------------------
# Quem são os clientes com TotalCharges ausente?
# ---------------------------------------------------------------
afetados = df_telco_limpo[df_telco_limpo["TotalCharges"].isna()]

display(afetados[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

print("Valores de tenure entre os afetados:", sorted(afetados["tenure"].unique().tolist()))
print("\nChurn entre os afetados:")
print(afetados["Churn"].value_counts().to_string())

# Se todos os afetados têm tenure = 0, o significado é claro:
# são clientes recém-chegados que ainda não receberam nenhuma cobrança acumulada.
# Ou seja, o "vazio" não é um erro de coleta: é a ausência de cobrança.

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


Valores de tenure entre os afetados: [0]

Churn entre os afetados:
Churn
No    11


In [ ]:
#OPÇÕES PARA AUSENTES DE TotalCharges:

#A) Preencher com 0:  Se todos tem tenure = 0 -> Mantém os 11 clientes, 
# de forma coerente com nenhuma cobrança realizada. (NOSSO CASO)

#B) Manter como Nan: Se quiser deixar explicitamente "sem dado", 
# não altera a média, pois elas ignoram os nulos e mantém os clientes.

#C) Remover as linhas: Se os valores fossem erro de coleta, 
# com isso se perdem 11 clientes (aproximadamente 0,16% da base).

In [ ]:
# ---------------------------------------------------------------
# Aplicação da decisão A (com confirmação)
# ---------------------------------------------------------------
# Confirma a premissa: TODOS os ausentes têm tenure == 0.
# Se alguma linha com TotalCharges ausente tiver tenure > 0, o assert falha e interrompe a execução,
# evitando que preenchamos com zero um dado que na verdade está faltando.
assert (df_telco_limpo.loc[df_telco_limpo["TotalCharges"].isna(), "tenure"] == 0).all(), (
    "Há TotalCharges ausente em cliente com tenure > 0. Revise a decisão antes de preencher."
)

df_telco_limpo["TotalCharges"] = df_telco_limpo["TotalCharges"].fillna(0)

print("Nulos restantes em TotalCharges:", int(df_telco_limpo["TotalCharges"].isna().sum()))
# ESPERADO: 0

Nulos restantes em TotalCharges: 0


In [ ]:
# ---------------------------------------------------------------
# Validação e criação do Churn_Flag
# ---------------------------------------------------------------
# 1) Confirma que Churn só tem os valores esperados.
valores_churn = set(df_telco_limpo["Churn"].unique())
assert valores_churn == {"Yes", "No"}, f"Valores inesperados em Churn: {valores_churn}"

# 2) Cria a flag com um mapeamento EXPLÍCITO.
# Usamos .map({...}) em vez de comparar com "Yes" porque, se aparecer um valor
# que não está no dicionário, o resultado vira NaN e o .astype(int) abaixo acusa o erro
# (em vez de classificar silenciosamente como "não cancelou").
df_telco_limpo["Churn_Flag"] = df_telco_limpo["Churn"].map({"Yes": 1, "No": 0}).astype(int)

display(df_telco_limpo[["customerID", "Churn", "Churn_Flag"]].head())

,customerID,Churn,Churn_Flag
0,7590-VHVEG,No,0
1,5575-GNVDE,No,0
2,3668-QPYBK,Yes,1
3,7795-CFOCW,No,0
4,9237-HQITU,Yes,1


In [ ]:
# ---------------------------------------------------------------
# SeniorCitizen em formato textual
# ---------------------------------------------------------------
# POR QUÊ: SeniorCitizen é 0/1, mas as demais colunas "sim/não" usam "Yes"/"No".
# Criamos uma versão textual para padronizar legendas e filtros.
# Mantemos a coluna numérica original, a nova é apenas uma coluna ADICIONAL.
df_telco_limpo["SeniorCitizen_Label"] = df_telco_limpo["SeniorCitizen"].map({1: "Yes", 0: "No"})

assert df_telco_limpo["SeniorCitizen_Label"].notna().all(), "SeniorCitizen tem valores fora de 0/1."
print(df_telco_limpo["SeniorCitizen_Label"].value_counts().to_string())

SeniorCitizen_Label
No     5901
Yes    1142


In [ ]:
# ---------------------------------------------------------------
# Comparação "antes x depois"
# ---------------------------------------------------------------
# 1) Nenhuma linha foi perdida.
assert len(df_telco_limpo) == len(df_telco), "A limpeza alterou a quantidade de linhas."

# 2) customerID continua único.
assert df_telco_limpo["customerID"].is_unique, "customerID não é único."

# 3) Não restam nulos em nenhuma coluna.
assert df_telco_limpo.isna().sum().sum() == 0, "Ainda existem valores nulos."

print("Linhas  : antes =", len(df_telco), "| depois =", len(df_telco_limpo))
print("Colunas : antes =", df_telco.shape[1], "| depois =", df_telco_limpo.shape[1],
      "(+2 esperadas: Churn_Flag e SeniorCitizen_Label)")

# 4) O dado bruto continua intacto: TotalCharges ainda é texto lá.
print("\nTipo de TotalCharges no BRUTO  :", df_telco["TotalCharges"].dtype, "(deve continuar como object)")
print("Tipo de TotalCharges no LIMPO  :", df_telco_limpo["TotalCharges"].dtype, "(deve ser float64)")

print("\n--- Estrutura final do DataFrame limpo ---")
df_telco_limpo.info()

Linhas  : antes = 7043 | depois = 7043
Colunas : antes = 21 | depois = 23 (+2 esperadas: Churn_Flag e SeniorCitizen_Label)

Tipo de TotalCharges no BRUTO  : object (deve continuar como object)
Tipo de TotalCharges no LIMPO  : float64 (deve ser float64)

--- Estrutura final do DataFrame limpo ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customerID           7043 non-null   object 
 1   gender               7043 non-null   object 
 2   SeniorCitizen        7043 non-null   int64  
 3   Partner              7043 non-null   object 
 4   Dependents           7043 non-null   object 
 5   tenure               7043 non-null   int64  
 6   PhoneService         7043 non-null   object 
 7   MultipleLines        7043 non-null   object 
 8   InternetService      7043 non-null   object 
 9   OnlineSecurity       7043 non-null   obje

In [ ]:
# ---------------------------------------------------------------
# Conferência rápida de coerência: a primeira taxa de churn
# ---------------------------------------------------------------
# Média de uma coluna 0/1 = proporção de 1s = taxa de churn.
taxa_churn = df_telco_limpo["Churn_Flag"].mean()
n_churn = int(df_telco_limpo["Churn_Flag"].sum())

print(f"Clientes que cancelaram: {n_churn} de {len(df_telco_limpo)}")
print(f"Taxa de churn geral    : {taxa_churn:.2%}")

# ESPERADO (base clássica): 1869 de 7043, ou seja, ~26,54%.
# Isto é só uma conferência de sanidade e não uma análise de indicadores


Clientes que cancelaram: 1869 de 7043
Taxa de churn geral    : 26.54%


In [ ]:
# ---------------------------------------------------------------
# Verificando pré-requisitos para execução do código
# ---------------------------------------------------------------
# Toda esta etapa usa o df_telco_limpo, então vamos verificar
#  a existência dele e das Churn_Flag
try:
    df_telco_limpo
except NameError:
    raise NameError(
        "df_telco_limpo não existe. Execute as células da Etapa 3 (de cima para baixo) antes desta."
    )

assert "Churn_Flag" in df_telco_limpo.columns, "Churn_Flag não foi criada. Rode a célula 3.8."

print("Pré-requisitos OK:", len(df_telco_limpo), "clientes prontos para análise.")

Pré-requisitos OK: 7043 clientes prontos para análise.


In [ ]:
# ---------------------------------------------------------------
# Criando os indicadores principais
# ---------------------------------------------------------------
total_clientes = len(df_telco_limpo)                          # quantas linhas = quantos clientes
clientes_churn = int(df_telco_limpo["Churn_Flag"].sum())      # somar uma coluna 0/1 = contar os 1
taxa_churn = clientes_churn / total_clientes                  # denominador = TODOS os clientes analisados
cobranca_media = df_telco_limpo["MonthlyCharges"].mean()      # média da cobrança mensal

print(f"Total de clientes       : {total_clientes}")
print(f"Clientes que cancelaram : {clientes_churn}")
print(f"Taxa de churn           : {taxa_churn:.2%}")
print(f"Cobrança mensal média   : {cobranca_media:.2f}")

# Estas variáveis serão reutilizadas nas próximas células (ex.: a linha de média
# geral nos gráficos) e depois irão virar os cartões do dashboard.
# ESPERADO: 7043 · 1869 · ~26,54% · ~64,76.
# Se o seu resultado for bem diferente, volte à Etapa 3 e revise a limpeza.

Total de clientes       : 7043
Clientes que cancelaram : 1869
Taxa de churn           : 26.54%
Cobrança mensal média   : 64.76


In [ ]:
# ---------------------------------------------------------------
# Distribuição de Churn (quantos "Yes" e quantos "No")
# ---------------------------------------------------------------
distribuicao = df_telco_limpo["Churn"].value_counts().to_frame("clientes")
distribuicao["percentual"] = (distribuicao["clientes"] / total_clientes * 100).round(1)
display(distribuicao)

# Isso mostra que a base é DESBALANCEADA: bem mais clientes
# permaneceram do que cancelaram (~73% x ~27% na base clássica).
# Isso é um contexto importante para ler qualquer gráfico e possivelmente
# montar um modelo preditivo.

,clientes,percentual
Churn,,
No,5174,73.5
Yes,1869,26.5


In [ ]:
# ---------------------------------------------------------------
# Função reutilizável: churn por grupo
# ---------------------------------------------------------------
def resumo_churn(df, coluna):
    """
    Calcula o churn de cada categoria de uma coluna.

    Parâmetros
    ----------
    df     : DataFrame com as colunas "customerID" e "Churn_Flag".
    coluna : nome da coluna usada para separar os grupos (ex.: "Contract").

    Retorna um DataFrame com, por grupo:
      clientes        -> quantos clientes há no grupo (o TAMANHO do grupo)
      cancelaram      -> quantos deles cancelaram
      taxa_churn      -> cancelaram / clientes, em fração (0.25 = 25%)
      dif_vs_geral_pp -> diferença da taxa do grupo para a taxa da base completa,
                         em pontos percentuais (p.p.). Positivo = acima da média.
    """
    # Taxa da base COMPLETA, usada como referência de comparação.

    taxa_geral = df_telco_limpo["Churn_Flag"].mean()

    resumo = (
        df.groupby(coluna, observed=True)          # separa em grupos pela coluna escolhida
        .agg(
            clientes=("customerID", "count"),      # conta clientes do grupo
            cancelaram=("Churn_Flag", "sum"),      # soma dos 1 = quem cancelou
            taxa_churn=("Churn_Flag", "mean"),     # média de 0/1 = proporção de churn
        )
        .sort_values("taxa_churn", ascending=False)  # maior taxa primeiro
    )

    resumo["dif_vs_geral_pp"] = (resumo["taxa_churn"] - taxa_geral) * 100

    # Trava de segurança: a soma dos grupos deve dar o total de linhas recebidas.
    # Se não der, a coluna tem nulos (o groupby descarta nulos em silêncio).
    assert resumo["clientes"].sum() == len(df), f"Há valores nulos em '{coluna}': grupos não somam o total."

    return resumo


def mostra_resumo(resumo):
    """Exibe o resumo com a taxa em % (só formatação; os valores continuam em fração)."""
    try:
        display(resumo.style.format({"taxa_churn": "{:.1%}", "dif_vs_geral_pp": "{:+.1f}"}))
    except ImportError:
        # .style exige a biblioteca jinja2; sem ela, mostra a tabela sem formatação
        display(resumo)

In [ ]:
# ---------------------------------------------------------------
# Três comparações com a mesma função
# ---------------------------------------------------------------
dimensoes = {
    "Contract": "Tipo de contrato",
    "InternetService": "Tipo de internet",
    "PaymentMethod": "Forma de pagamento",
}

for coluna, rotulo in dimensoes.items():
    print(f"\n=== Churn por {rotulo} ({coluna}) ===")
    mostra_resumo(resumo_churn(df_telco_limpo, coluna))

# O dicionário "dimensoes" será reaproveitado nos gráficos e no rascunho
# dos achados. Para analisar outra coluna depois, basta acrescentar uma linha nele.
#
# ESPERADO (base clássica, valores aproximados):
#   Contract        -> Month-to-month perto de 43%; One year perto de 11%; Two year perto de 3%
#   InternetService -> Fiber optic perto de 42%; DSL perto de 19%; No perto de 7%
#   PaymentMethod   -> Electronic check perto de 45%; os demais entre 15% e 19%
# Se os padrões forem bem diferentes, volte à Etapa 3 e revise.
#
# ATENÇÃO: o que estas tabelas mostram é ASSOCIAÇÃO. Os grupos se sobrepõem (quem tem fibra
# também tende a ter outro tipo de contrato e de pagamento), então não dá para dizer que uma
# característica "causa" o churn.

# Dica de leitura: dif_vs_geral_pp = +15 significa que o grupo está 15 pontos percentuais
# ACIMA da taxa geral (ex.: 41,5% contra 26,5%). Para não confundir p.p. com "15% maior".


=== Churn por Tipo de contrato (Contract) ===


,clientes,cancelaram,taxa_churn,dif_vs_geral_pp
Contract,,,,
Month-to-month,3875,1655,42.7%,+16.2
One year,1473,166,11.3%,-15.3
Two year,1695,48,2.8%,-23.7



=== Churn por Tipo de internet (InternetService) ===


,clientes,cancelaram,taxa_churn,dif_vs_geral_pp
InternetService,,,,
Fiber optic,3096,1297,41.9%,+15.4
DSL,2421,459,19.0%,-7.6
No,1526,113,7.4%,-19.1



=== Churn por Forma de pagamento (PaymentMethod) ===


,clientes,cancelaram,taxa_churn,dif_vs_geral_pp
PaymentMethod,,,,
Electronic check,2365,1071,45.3%,+18.7
Mailed check,1612,308,19.1%,-7.4
Bank transfer (automatic),1544,258,16.7%,-9.8
Credit card (automatic),1522,232,15.2%,-11.3


In [ ]:
# ---------------------------------------------------------------
# Tenure: quem cancelou x quem ficou
# ---------------------------------------------------------------
display(df_telco_limpo.groupby("Churn")["tenure"].describe().round(2))

# Dica de leitura: como ler as colunas? count = nº de clientes; mean = média; std = dispersão;
# min/max = extremos; 25%/50%/75% = quartis (50% é a MEDIANA).

# ESPERADO (base clássica): quem cancelou tem tenure bem menor (média por volta de 18 meses
# contra 37 a 38 de quem ficou). Ou seja, o cancelamento está ASSOCIADO a clientes mais novos.

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,37.57,24.11,0.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.98,19.53,1.0,2.0,10.0,29.0,72.0


In [ ]:
# ---------------------------------------------------------------
# Cobrança mensal: quem cancelou x quem ficou
# ---------------------------------------------------------------
display(df_telco_limpo.groupby("Churn")["MonthlyCharges"].agg(["count", "mean", "median"]))

# Compare média E mediana. Se forem bem diferentes, a distribuição tem "cauda"
# (poucos valores muito altos ou muito baixos puxando a média) e a média sozinha engana.

# ESPERADO (base clássica): a cobrança mensal tende a ser MAIOR entre quem cancelou.
# IMPORTANTE: isso não quer dizer que "preço alto causa churn", pode refletir o tipo de
# serviço contratado (ex.: fibra é mais cara). Anote como hipótese, não como conclusão.

,count,mean,median
Churn,,,
No,5174,61.265124,64.425
Yes,1869,74.441332,79.650


In [ ]:
# ---------------------------------------------------------------
# Faixas de tenure: taxa de churn por tempo de casa
# ---------------------------------------------------------------
# pd.cut transforma um número (tenure) em categorias (faixas).
# Importante: por que o primeiro limite é -1 e não 0? Porque o pd.cut usa intervalos
# "abertos à esquerda" ((a, b]). Com limite 0, quem tem tenure = 0 ficaria FORA de todas as
# faixas (viraria NaN). Com -1, a primeira faixa é (-1, 12] e inclui o 0.
df_telco_limpo["faixa_tenure"] = pd.cut(
    df_telco_limpo["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 meses", "13-24 meses", "25-48 meses", "49-72 meses"],
)

# Trava de segurança: nenhum cliente pode ter ficado sem faixa
assert df_telco_limpo["faixa_tenure"].notna().all(), "Há clientes fora das faixas: revise os limites (bins)."

# .sort_index() mantém as faixas na ORDEM DO TEMPO (e não na ordem da taxa)
mostra_resumo(resumo_churn(df_telco_limpo, "faixa_tenure").sort_index())

# ATENÇÃO: criamos uma coluna nova (faixa_tenure) no df_telco_limpo. Ela é uma coluna derivada
# para análise; o app Streamlit vai precisar recriá-la ao carregar os dados.

# ESPERADO (base clássica): a faixa 0-12 meses tem a maior taxa e ela diminui conforme o tempo
# de casa aumenta.

,clientes,cancelaram,taxa_churn,dif_vs_geral_pp
faixa_tenure,,,,
0-12 meses,2186,1037,47.4%,+20.9
13-24 meses,1024,294,28.7%,+2.2
25-48 meses,1594,325,20.4%,-6.1
49-72 meses,2239,213,9.5%,-17.0


In [ ]:
# ---------------------------------------------------------------
# Importação do Plotly e função de gráfico de barras para taxas
# ---------------------------------------------------------------
try:
    import plotly.express as px
except ImportError:
    raise ImportError(
        "A biblioteca plotly não está instalada. No terminal, rode: pip install plotly\n"
        "Depois reinicie o kernel do notebook (menu Kernel > Restart) e execute as células de novo."
    )


def grafico_taxa(resumo, coluna, titulo):
    # Gráfico de barras da taxa de churn por grupo, com a média geral como linha de referência.
    dados = resumo.reset_index()   # o nome do grupo sai do índice e vira coluna
    fig = px.bar(
        dados,
        x=coluna,
        y="taxa_churn",
        text_auto=".1%",                          # rótulo em % em cima de cada barra
        hover_data=["clientes", "cancelaram"],    # ao passar o mouse, mostra o tamanho do grupo
        title=titulo,
        labels={"taxa_churn": "Taxa de churn"},
    )
    fig.update_yaxes(tickformat=".0%")            # eixo Y em porcentagem
    fig.add_hline(y=taxa_churn, line_dash="dash", annotation_text="média geral")
    return fig

# Dica de leitura: a linha tracejada é a taxa geral (4.1). Barra acima dela = grupo com churn
# acima da média; barra abaixo = grupo abaixo da média.
# ATENÇÃO: o tamanho do grupo aparece no "hover" (passe o mouse).

In [ ]:
# ---------------------------------------------------------------
# Barras: churn por contrato, internet e pagamento
# ---------------------------------------------------------------
for coluna, rotulo in dimensoes.items():
    fig = grafico_taxa(resumo_churn(df_telco_limpo, coluna), coluna, f"Taxa de churn por {rotulo.lower()}")
    fig.show()

# ESPERADO: três gráficos de barras, cada um com a linha tracejada da média geral.
# AUXÍLIO DE EXECUÇÃO: se nenhum gráfico aparecer, reinicie o kernel e rode de novo a célula do import.

In [ ]:
# ---------------------------------------------------------------
# Tenure: barras por faixa e boxplot por situação
# ---------------------------------------------------------------
resumo_faixa = resumo_churn(df_telco_limpo, "faixa_tenure").sort_index()
grafico_taxa(resumo_faixa, "faixa_tenure", "Taxa de churn por faixa de tempo de casa").show()

px.box(
    df_telco_limpo,
    x="Churn",
    y="tenure",
    color="Churn",
    title="Tempo de permanência: quem cancelou x quem ficou",
    labels={"tenure": "Tempo de permanência (meses)", "Churn": "Cancelou?"},
).show()

# ---------------------------------------------------------------
# Guia rápido: Como ler um Boxplot
# ---------------------------------------------------------------
# - Linha do meio    : Mediana (o valor exato que divide a turma ao meio).
# - Caixa (retângulo): A metade central dos dados (onde fica a maior parte dos clientes).
# - "Bigodes" (linhas) : As linhas que se estendem para cima e para baixo, 
#                      delimitando a faixa dos valores normais e esperados.
# - Pontos isolados  : Outliers (casos raros ou extremos que fugiram muito da curva).

# ESPERADO: a caixa de quem cancelou ("Yes") fica bem mais abaixo da de quem ficou ("No").

In [ ]:
# ---------------------------------------------------------------
# Cobrança mensal: distribuição por situação
# ---------------------------------------------------------------
px.histogram(
    df_telco_limpo,
    x="MonthlyCharges",
    color="Churn",
    barmode="overlay",       # sobrepõe os dois grupos para facilitar a comparação
    opacity=0.6,
    histnorm="percent",      # cada grupo soma 100%
    nbins=30,
    title="Distribuição da cobrança mensal: quem cancelou x quem ficou",
    labels={"MonthlyCharges": "Cobrança mensal"},
).show()

# Por que histnorm="percent"? Porque há bem mais clientes que ficaram do que
# clientes que cancelaram. Sem normalizar, as barras de "No" seriam sempre maiores só por causa
# do tamanho do grupo, e a comparação das FORMAS ficaria impossível.

In [ ]:
# ---------------------------------------------------------------
# Maior e menor taxa de cada dimensão analisada
# ---------------------------------------------------------------
dimensoes_achados = {**dimensoes, "faixa_tenure": "Faixa de tenure"}

for coluna, rotulo in dimensoes_achados.items():
    resumo = resumo_churn(df_telco_limpo, coluna)   # já vem ordenado: maior taxa no topo
    maior, menor = resumo.iloc[0], resumo.iloc[-1]
    diferenca_pp = (maior["taxa_churn"] - menor["taxa_churn"]) * 100

    print(f"\n{rotulo}")
    print(f"  Maior taxa: {resumo.index[0]} -> {maior['taxa_churn']:.1%} "
          f"({int(maior['cancelaram'])} de {int(maior['clientes'])} clientes)")
    print(f"  Menor taxa: {resumo.index[-1]} -> {menor['taxa_churn']:.1%} "
          f"({int(menor['cancelaram'])} de {int(menor['clientes'])} clientes)")
    print(f"  Diferença : {diferenca_pp:.1f} p.p.")

# POR QUÊ: transforma as tabelas em um resumo de uma linha por dimensão, pronto para virar
# achado. Note que cada linha traz "X de Y clientes": é a ressalva de tamanho de grupo.
# ATENÇÃO: "maior taxa" não é "causa", só mostra que merece investigação.
# LEMBRANDO: um segmento merece investigação quando tem taxa bem acima da média E tamanho
# relevante. Taxa alta em grupo minúsculo é pouco confiável; grupo grande com taxa alta é
# onde uma ação de retenção teria mais alcance.


Tipo de contrato
  Maior taxa: Month-to-month -> 42.7% (1655 de 3875 clientes)
  Menor taxa: Two year -> 2.8% (48 de 1695 clientes)
  Diferença : 39.9 p.p.

Tipo de internet
  Maior taxa: Fiber optic -> 41.9% (1297 de 3096 clientes)
  Menor taxa: No -> 7.4% (113 de 1526 clientes)
  Diferença : 34.5 p.p.

Forma de pagamento
  Maior taxa: Electronic check -> 45.3% (1071 de 2365 clientes)
  Menor taxa: Credit card (automatic) -> 15.2% (232 de 1522 clientes)
  Diferença : 30.0 p.p.

Faixa de tenure
  Maior taxa: 0-12 meses -> 47.4% (1037 de 2186 clientes)
  Menor taxa: 49-72 meses -> 9.5% (213 de 2239 clientes)
  Diferença : 37.9 p.p.


### Registro dos achados

| # | Pergunta | Métrica | Grupos comparados | Resultado observado | Ressalvas |
|---|---|---|---|---|---|
| 1 | Qual é a proporção de clientes que cancelaram? | Taxa de churn geral | Base completa | 1.869 de 7.043 clientes (26,5%) | Base desbalanceada (cerca de 73% x 27%) |
| 2 | A taxa varia por contrato? | Taxa de churn | Mensal x 1 ano x 2 anos | Mensal 42,7% · 1 ano 11,3% · 2 anos 2,8% | Grupos grandes (3.875 · 1.473 · 1.695). Associação, não causa |
| 3 | Há diferença por internet? | Taxa de churn | DSL x Fibra x Sem internet | Fibra 41,9% · DSL 19,0% · Sem internet 7,4% | Grupos de 3.096 · 2.421 · 1.526. Fibra também se associa a cobrança maior e a contrato mensal |
| 4 | Há diferença por forma de pagamento? | Taxa de churn | Cheque eletrônico x demais formas | Cheque eletrônico 45,3% · cheque pelo correio 19,1% · transferência 16,7% · cartão 15,2% | Grupos de 2.365 · 1.612 · 1.544 · 1.522. Pode acompanhar outros fatores (contrato) |
| 5 | O tenure difere entre quem cancelou e quem ficou? | Média e mediana de tenure | Cancelou x Permaneceu | Cancelou: média 18,0 e mediana 10 meses. Permaneceu: média 37,6 e mediana 38. Faixa 0-12 meses com cerca de 47% de churn, contra cerca de 10% em 49-72 | Cancelamento associado a clientes mais novos. Valores das faixas aproximados |
| 6 | A cobrança mensal difere entre os grupos? | Média e mediana | Cancelou x Permaneceu | Cancelou: média 74,4 e mediana 79,7. Permaneceu: média 61,3 e mediana 64,4 | Pode refletir o tipo de serviço (a fibra é mais cara) |
| 7 | Quais segmentos merecem investigação? | Taxa e tamanho do grupo | Contrato mensal, fibra óptica, cheque eletrônico, 0-12 meses | Os quatro apresentam taxa bem acima da média geral (26,5%), em grupos grandes | Os segmentos se sobrepõem; investigar combinações antes de qualquer ação (hipótese) |

- ___

### Critério de conclusão da EDA (do guia do projeto)
- [ ] Os indicadores principais foram calculados e conferidos.
- [ ] As comparações por contrato, internet, pagamento e tenure foram feitas.
- [ ] Em cada comparação, o tamanho dos grupos foi verificado antes de interpretar a taxa.
- [ ] Os gráficos relevantes foram gerados e escolhidos conforme as perguntas feitas.

---

In [ ]:
# ---------------------------------------------------------------
# Verificação de sobreposição: a diferença resiste dentro do mesmo contrato?
# ---------------------------------------------------------------
# Contrato, internet e pagamento andam juntos. Cruzar as colunas mostra se a
# diferença permanece quando o tipo de contrato é o mesmo.
MIN_CLIENTES_CELULA = 100   # combinações menores ficam em branco (NaN)

def cruza_taxa(df, coluna_linhas, coluna_colunas):
    taxa = df.pivot_table(index=coluna_linhas, columns=coluna_colunas, values="Churn_Flag",
                          aggfunc="mean", observed=True) * 100
    n = df.pivot_table(index=coluna_linhas, columns=coluna_colunas, values="customerID",
                       aggfunc="count", observed=True)
    taxa = taxa.where(n >= MIN_CLIENTES_CELULA).round(1)
    return taxa, n

for coluna in ["InternetService", "PaymentMethod", "faixa_tenure"]:
    taxa, n = cruza_taxa(df_telco_limpo, "Contract", coluna)
    print(f"\n=== Taxa de churn (%) por Contract x {coluna} ===")
    display(taxa)
    print("Clientes em cada combinação:")
    display(n)

# Célula em branco = menos de 100 clientes (não tire conclusão dela).
# Mesmo que a diferença permaneça, continua sendo associação, não causa.


=== Taxa de churn (%) por Contract x InternetService ===


InternetService,DSL,Fiber optic,No
Contract,,,
Month-to-month,32.2,54.6,18.9
One year,9.3,19.3,2.5
Two year,1.9,7.2,0.8


Clientes em cada combinação:


InternetService,DSL,Fiber optic,No
Contract,,,
Month-to-month,1223,2128,524
One year,570,539,364
Two year,628,429,638



=== Taxa de churn (%) por Contract x PaymentMethod ===


...

PaymentMethod,Bank transfer (automatic),Credit card (automatic),Electronic check,Mailed check
Contract,,,,
Month-to-month,589,543,1850,893
One year,391,398,347,337
Two year,564,581,168,382



=== Taxa de churn (%) por Contract x faixa_tenure ===


faixa_tenure,0-12 meses,13-24 meses,25-48 meses,49-72 meses
Contract,,,,
Month-to-month,51.4,37.7,32.9,26.0
One year,10.5,8.1,10.6,12.9
Two year,NaN,NaN,2.2,3.3


Clientes em cada combinação:


faixa_tenure,0-12 meses,13-24 meses,25-48 meses,49-72 meses
Contract,,,,
Month-to-month,1994,737,802,342
One year,124,197,518,634
Two year,68,90,274,1263


## Insights

### Insight 1: Contrato mensal concentra mais cancelamentos
- **Evidência observada:** Contrato mensal tem churn de 42,7% (3.875 clientes), contra 2,8% nos contratos de 2 anos (1.695 clientes).
- **Interpretação prudente:** O contrato mensal está associado a maior churn. Os dados não mostram o motivo: pode refletir o perfil do cliente, o preço ou apenas a facilidade de sair.
- **Possível ação a testar:** Testar uma oferta de migração para contrato anual entre clientes do contrato mensal.
- **Como avaliar:** Comparar o churn de quem recebeu a oferta com o de um grupo parecido que não recebeu, ao longo de alguns meses.
- **Fonte:** Gráfico "Taxa de churn por tipo de contrato".
- **Ressalva (Código A):** A diferença permanece dentro de cada tipo de internet. Mensal contra 2 anos: DSL 32,2% contra 1,9%; fibra 54,6% contra 7,2%; sem internet 18,9% contra 0,8%. Fatores não medidos, como o motivo do cancelamento, ainda podem influenciar.

### Insight 2: Fibra óptica tem churn maior que DSL
- **Evidência observada:** Churn de 41,9% na fibra (3.096 clientes) e 19,0% no DSL (2.421 clientes). Cobrança mensal média: 93,93 na fibra e 60,21 no DSL.
- **Interpretação prudente:** A fibra está associada a maior churn, mas também a cobrança mais alta. Os dados não separam o efeito do serviço do efeito do preço ou do contrato.
- **Possível ação a testar:** Investigar a experiência desses clientes (qualidade, suporte, preço percebido) por pesquisa de satisfação ou entrevistas.
- **Como avaliar:** Comparar satisfação e churn dos clientes de fibra antes e depois de qualquer melhoria.
- **Fonte:** Gráfico "Taxa de churn por tipo de internet".
- **Ressalva (Código A):** A fibra tem churn maior em todos os contratos: 54,6% contra 32,2% no mensal, 19,3% contra 9,3% em 1 ano, 7,2% contra 1,9% em 2 anos. Ela também concentra mais contratos mensais (cerca de 69% dos clientes de fibra, contra 51% no DSL). Sem o motivo do cancelamento, não dá para atribuir a diferença ao serviço.

### Insight 3: O cancelamento se concentra no início do relacionamento
- **Evidência observada:** Na faixa de 0-12 meses o churn é de cerca de 47,4% (2.186 clientes), contra cerca de 9,5% na faixa de 49-72 meses (2.239 clientes).
- **Interpretação prudente:** Clientes mais novos cancelam mais. Os dados não explicam por quê: mostram apenas associação com o início do relacionamento.
- **Possível ação a testar:** Avaliar ações de acompanhamento no primeiro ano (boas-vindas, contato proativo, suporte inicial), começando pelos clientes novos de contrato mensal.
- **Como avaliar:** Medir o churn dos novos clientes que receberam a ação contra um grupo parecido que não recebeu.
- **Fonte:** Gráfico "Taxa de churn por tempo como cliente".
- **Ressalva (Código A):** A queda com o tempo aparece no contrato mensal (51,4% → 37,7% → 32,9% → 26,0%), mas não no de 1 ano (10,5%, 8,1%, 10,6%, 12,9%). No de 2 anos só há clientes suficientes nas duas últimas faixas (2,2% e 3,3%). Além disso, 91% dos clientes de 0-12 meses têm contrato mensal (1.994 de 2.186). As faixas de tempo foram escolhidas por mim.

### Insight 4: Cheque eletrônico aparece associado a mais churn
- **Evidência observada:** Churn de 45,3% entre quem paga por cheque eletrônico (2.365 clientes), contra 15,2% a 19,1% nas outras formas de pagamento.
- **Interpretação prudente:** A associação existe, mas a forma de pagamento pode estar acompanhando outro fator, como o tipo de contrato. É um ponto para investigar, não uma conclusão.
- **Possível ação a testar:** Investigar se há atrito no pagamento e testar incentivos ao pagamento automático.
- **Como avaliar:** Comparar o churn de quem migrou para pagamento automático com o de um grupo parecido que não migrou.
- **Fonte:** Gráfico "Taxa de churn por forma de pagamento".
- **Ressalva (Código A):** Dentro do contrato mensal, o cheque eletrônico tem 53,7% (1.850 clientes), contra 31,6% a 34,1% nas outras formas. A diferença é de cerca de 20 pontos percentuais, menor que na base toda (cerca de 27), mas continua. Isso acontece porque 78% de quem usa cheque eletrônico está no contrato mensal (1.850 de 2.365), contra 36% a 55% nas outras formas.